# Линейный и нелинейный маятник: период и сходимость методов

## 1. Уравнения и начальные условия

Угол $\theta$ отсчитывается от вертикали вниз, $v=\dot\theta$ — угловая скорость. Для маятника без трения
$$\dot\theta=v,\qquad \dot v=-\frac{g}{\ell}\sin\theta.$$
Линейная модель получается при замене $\sin\theta$ на $\theta$:
$$\dot v=-\omega_0^2\theta,\qquad \omega_0=\sqrt{g/\ell}.$$

В основных расчётах $g=9.81$ м/с², $\ell=1$ м, масса груза $m=1$ кг. Маятник отпускается из крайнего положения: $\theta(0)=A$, $v(0)=0$. Углы внутри программы выражены в радианах; градусы используются для задания амплитуд и подписей.

Точное линейное решение:
$$\theta_{\rm lin}(t)=A\cos(\omega_0t),\qquad v_{\rm lin}(t)=-A\omega_0\sin(\omega_0t),\qquad T_0=\frac{2\pi}{\omega_0}.$$
Для выбранной длины $T_0=2.006067$ с. Его использую как единицу времени на части графиков: отметка $t/T_0=5$ означает пять **линейных** периодов.

В явном методе Эйлера обе компоненты обновляются по старому состоянию:
$$\theta_{n+1}=\theta_n+hv_n,\qquad v_{n+1}=v_n-h\omega_0^2\theta_n.$$
Для нелинейной модели во второй формуле вместо $\theta_n$ стоит $\sin\theta_n$. RK4 использует четыре оценки правой части:
$$k_1=f(y_n),\quad k_2=f(y_n+hk_1/2),\quad k_3=f(y_n+hk_2/2),\quad k_4=f(y_n+hk_3),$$
$$y_{n+1}=y_n+\frac h6(k_1+2k_2+2k_3+k_4),\qquad y=(\theta,v).$$

В функции интегрирования задаётся число шагов $N$. Сетка равномерна, её фактический шаг равен $h=t_{\rm end}/N$. Это позволяет сравнивать решения на одном интервале и точно делить шаг пополам.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import ellipk, ellipj
from scipy.optimize import brentq
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter, writers
from IPython.display import HTML, Image, display
from pathlib import Path
import subprocess

OUTPUT = Path("pendulum_results")
for directory in [OUTPUT, OUTPUT/"figures", OUTPUT/"data", OUTPUT/"media"]:
    directory.mkdir(exist_ok=True)
plt.rcParams.update({"font.size":11,"axes.spines.top":False,"axes.spines.right":False})
g, length, mass = 9.81, 1.0, 1.0
omega0 = np.sqrt(g/length)
T0 = 2*np.pi/omega0
colors = {"Эйлер":"#c34a36","RK4":"#2166ac"}


def rhs(y, nonlinear=True, ell=length):
    theta, velocity = y
    return np.array([velocity, -g/ell*(np.sin(theta) if nonlinear else theta)])


def rk4_step(y, h, nonlinear=True, ell=length):
    k1 = rhs(y, nonlinear, ell)
    k2 = rhs(y+h*k1/2, nonlinear, ell)
    k3 = rhs(y+h*k2/2, nonlinear, ell)
    k4 = rhs(y+h*k3, nonlinear, ell)
    return y+h*(k1+2*k2+2*k3+k4)/6


def integrate(amplitude, duration, nsteps, method="RK4", nonlinear=True, ell=length):
    t = np.linspace(0,duration,nsteps+1)
    h = duration/nsteps
    y = np.empty((nsteps+1,2))
    y[0] = [amplitude,0.0]
    for n in range(nsteps):
        if method == "Эйлер":
            y[n+1] = y[n]+h*rhs(y[n],nonlinear,ell)
        elif method == "RK4":
            y[n+1] = rk4_step(y[n],h,nonlinear,ell)
        else:
            raise ValueError(method)
    return t,y


def exact_linear(t, amplitude, ell=length):
    w = np.sqrt(g/ell)
    return np.array([amplitude*np.cos(w*np.asarray(t)),
                     -amplitude*w*np.sin(w*np.asarray(t))]).T


def exact_nonlinear(t, amplitude, ell=length):
    k = np.sin(amplitude/2)
    sn,cn,_,_ = ellipj(ellipk(k*k)-np.sqrt(g/ell)*np.asarray(t),k*k)
    return np.array([2*np.arcsin(k*sn),-2*k*np.sqrt(g/ell)*cn]).T


def exact_period(amplitude, ell=length):
    return 4*np.sqrt(ell/g)*ellipk(np.sin(amplitude/2)**2)


def energies(y, nonlinear=True, ell=length):
    theta,velocity = np.asarray(y).T
    U = mass*g*ell*(1-np.cos(theta) if nonlinear else theta**2/2)
    K = mass*ell**2*velocity**2/2
    return U,K


def numerical_period(amplitude, h):
    y = np.array([amplitude,0.0])
    for n in range(200000):
        next_y = rk4_step(y,h)
        if y[0]>0 and next_y[0]<=0:
            def crossing(s):
                return ((2*s**3-3*s**2+1)*y[0]+(s**3-2*s**2+s)*h*y[1]
                       +(-2*s**3+3*s**2)*next_y[0]+(s**3-s**2)*h*next_y[1])
            s = brentq(crossing,0,1,xtol=1e-14)
            return 4*(n+s)*h
        y = next_y
    raise RuntimeError("Пересечение нуля не найдено")


def show_table(frame, filename, formatters):
    frame.to_csv(OUTPUT/"data"/f"{filename}.csv",index=False)
    display(HTML(frame.to_html(index=False,formatters=formatters,border=0,na_rep="—")))


def show_figure(fig, filename):
    fig.savefig(OUTPUT/"figures"/f"{filename}.png",dpi=150)
    plt.show()

## 2. Проверка методов на линейном маятнике

Возьму $A=0.1$ рад (около $5.73$ градусов) и интервал $0\le t\le5T_0$. На каждом графике сравниваются три шага: $T_0/100$, $T_0/200$ и $T_0/400$. Пунктиром показано точное решение. Раздельные панели нужны здесь для сравнения **методов**

In [ ]:
small_amplitude = 0.1
linear_duration = 5*T0
resolutions = [100,200,400,800,1600]
linear_results = {(method,M):integrate(small_amplitude,linear_duration,5*M,method,False)
                  for method in ["Эйлер","RK4"] for M in resolutions}
t_fine = np.linspace(0,linear_duration,3001)
linear_exact = exact_linear(t_fine,small_amplitude)
fig,axes = plt.subplots(1,2,figsize=(11,4.2),constrained_layout=True)
for ax,method in zip(axes,["Эйлер","RK4"]):
    for M in resolutions[:3]:
        t,y = linear_results[method,M]
        ax.plot(t/T0,y[:,0],lw=1.2,label=f"h = T₀/{M}")
    ax.plot(t_fine/T0,linear_exact[:,0],"k--",lw=1.2,label="Точное решение")
    ax.set(title=method,xlabel="Время t/T₀",ylabel="Угол θ, рад")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=9)
show_figure(fig,"01_linear_angles")
fig,axes = plt.subplots(1,2,figsize=(11,4.2),constrained_layout=True)
for ax,method in zip(axes,["Эйлер","RK4"]):
    for M in resolutions[:3]:
        _,y = linear_results[method,M]
        ax.plot(y[:,0],y[:,1],lw=1.2,label=f"h = T₀/{M}")
    ax.plot(linear_exact[:,0],linear_exact[:,1],"k--",lw=1.2,label="Точное решение")
    ax.set(title=method,xlabel="Угол θ, рад",ylabel="Угловая скорость v, рад/с")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=9)
show_figure(fig,"02_linear_phase")

**Рисунки 1 и 2.** Сверху показан угол во времени, ниже — фазовая траектория $(\theta,v)$. У Эйлера амплитуда увеличивается, поэтому фазовая траектория раскручивается наружу. Уменьшение шага ослабляет этот эффект. RK4 на выбранных шагах почти совпадает с точным решением, поэтому его ошибку удобнее оценивать численно.

Для сравнения использую максимальную ошибку угла по узлам сетки:
$$\varepsilon_\theta(h)=\max_n|\theta_n-\theta_{\rm exact}(t_n)|.$$
Оценка порядка между соседними сетками:
$$p=\frac{\ln[\varepsilon_\theta(h)/\varepsilon_\theta(h/2)]}{\ln2}.$$
Столбец с порядком относится к сравнению текущей строки с предыдущей для того же метода. Для самой грубой сетки такой оценки нет.

In [ ]:
linear_rows = []
for method in ["Эйлер","RK4"]:
    previous = None
    for M in resolutions:
        t,y = linear_results[method,M]
        error = np.max(np.abs(y[:,0]-exact_linear(t,small_amplitude)[:,0]))
        U,K = energies(y,False)
        relative_energy = np.max(np.abs((U+K)/(U[0]+K[0])-1))*100
        p = np.nan if previous is None else np.log2(previous/error)
        linear_rows.append({"Метод":method,"Шаг h, с":t[1]-t[0],
                            "Ошибка угла, рад":error,"Оценка порядка p":p,
                            "Ошибка энергии, %":relative_energy})
        previous = error
linear_table = pd.DataFrame(linear_rows)
show_table(linear_table,"linear_convergence",{
    "Шаг h, с":lambda x:f"{x:.7f}","Ошибка угла, рад":lambda x:f"{x:.3e}",
    "Оценка порядка p":lambda x:"—" if np.isnan(x) else f"{x:.3f}",
    "Ошибка энергии, %":lambda x:f"{x:.3e}"})

**Таблица 1.** Ошибки на интервале $[0,5T_0]$. Ошибка энергии — максимум $100|E_n/E_0-1|$. Запись `e-06` означает умножение на $10^{-6}$.

На последней паре сеток получено $p\approx1.045$ для Эйлера и $p\approx4.000$ для RK4. Это согласуется с первым и четвёртым глобальными порядками методов. На грубых сетках ошибка Эйлера может уменьшаться более чем вдвое: рост амплитуды ещё достаточно велик, и асимптотический режим $\varepsilon\sim Ch$ не установился.

### Что происходит с энергией

Для линейного уравнения нужно использовать согласованную с ним квадратичную потенциальную энергию:
$$U_{\rm lin}=\frac{mg\ell}{2}\theta^2,\qquad K=\frac{m\ell^2}{2}v^2,\qquad E=U+K.$$
Точная сумма постоянна, а $U$ и $K$ периодически переходят друг в друга. Подстановка формул Эйлера даёт
$$E_{n+1}=(1+\omega_0^2h^2)E_n.$$
Следовательно, при любом ненулевом фиксированном шаге этот метод увеличивает энергию. На фиксированном интервале уменьшение шага улучшает результат, но для долгого моделирования незатухающих колебаний явный Эйлер неудобен.

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(11,4.2),constrained_layout=True)
for ax,method in zip(axes,["Эйлер","RK4"]):
    t,y = linear_results[method,200]
    U,K = energies(y,False)
    ax.plot(t/T0,U,label="Потенциальная U")
    ax.plot(t/T0,K,label="Кинетическая K")
    ax.plot(t/T0,U+K,label="Полная E",color="#34785a")
    ax.axhline(U[0]+K[0],color="black",ls="--",lw=1,label="Точная полная энергия")
    ax.set(title=f"{method}, h = T₀/200",xlabel="Время t/T₀",ylabel="Энергия, Дж")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
show_figure(fig,"03_linear_energy")

**Рисунок 3.** Энергии линейного маятника при одном шаге $h=T_0/200$. У Эйлера увеличиваются максимумы обеих составляющих и их сумма. У RK4 сумма визуально постоянна; её небольшая численная ошибка приведена в таблице 1. Отсутствие заметного дрейфа на этом рисунке не означает точного сохранения энергии методом RK4.

## 3. Зависимость периода от амплитуды

Для нелинейного маятника потенциальная энергия равна $U=mg\ell(1-\cos\theta)$. При отпускании из угла $A$ закон сохранения энергии даёт
$$v^2=\frac{2g}{\ell}(\cos\theta-\cos A).$$
Из времени прохождения четверти колебания получается точный период
$$T(A)=4\sqrt{\frac{\ell}{g}}K(m),\qquad m=\sin^2\frac A2,$$
$$K(m)=\int_0^{\pi/2}\frac{d\varphi}{\sqrt{1-m\sin^2\varphi}}.$$
Здесь $m$ — **параметр** эллиптического интеграла, именно его принимает `scipy.special.ellipk`. Рассматриваются колебания с $0<A<\pi$.

Период численного решения определяю независимо от этой формулы: интегрирую RK4 до первого перехода угла через ноль вниз. При начальной скорости $v(0)=0$ это четверть периода. Внутри последнего шага ноль уточняется кубическим полиномом Эрмита по значениям угла и угловой скорости на концах. Затем $T_{\rm num}=4t_{\rm cross}$. Простое округление момента пересечения до узла сетки внесло бы дополнительную ошибку времени.

Проверю амплитуды от $1^\circ$ до $170^\circ$ при шаге RK4 $0.002$ с.

In [ ]:
amplitudes_deg = np.linspace(1,170,80)
amplitudes_rad = np.deg2rad(amplitudes_deg)
periods_exact = exact_period(amplitudes_rad)
periods_numeric = np.array([numerical_period(A,0.002) for A in amplitudes_rad])
period_relative_error = (periods_exact-T0)/periods_exact
fig,axes = plt.subplots(1,2,figsize=(11,4.2),constrained_layout=True)
axes[0].plot(amplitudes_deg,periods_exact,label="Нелинейная модель: точный период")
axes[0].plot(amplitudes_deg[::5],periods_numeric[::5],"o",ms=4,label="Нелинейная модель: RK4")
axes[0].axhline(T0,color="black",ls="--",label="Линейная модель")
axes[0].set(xlabel="Начальный угол A, градусы",ylabel="Период T, с",title="Период нелинейных колебаний")
axes[1].plot(amplitudes_deg,100*period_relative_error,color="#c34a36")
axes[1].axhline(1,color="black",ls="--",label="Допуск 1%")
axes[1].set(xlabel="Начальный угол A, градусы",ylabel="Ошибка линейного периода, %",title="Погрешность при умеренных амплитудах",xlim=(0,60),ylim=(0,8.5))
one_percent_angle = np.rad2deg(brentq(lambda A:1-T0/exact_period(A)-0.01,1e-7,np.pi-1e-7))
axes[1].plot(one_percent_angle,1,"ko",ms=4)
axes[1].annotate(f"{one_percent_angle:.2f}°",xy=(one_percent_angle,1),xytext=(one_percent_angle+9,2),arrowprops=dict(arrowstyle="->",color="0.3"))
for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
show_figure(fig,"04_period_amplitude")
period_angles = [5,10,20,30,45,60,90,120,150,170]
period_rows = []
for deg in period_angles:
    exact_T = float(exact_period(np.deg2rad(deg)))
    numeric_T = numerical_period(np.deg2rad(deg),0.002)
    period_rows.append({"Начальный угол, °":deg,"Точный период, с":exact_T,
                        "Период RK4, с":numeric_T,"Ошибка линейного периода, %":100*(exact_T-T0)/exact_T})
period_table = pd.DataFrame(period_rows)
show_table(period_table,"period_amplitude",{
    "Точный период, с":lambda x:f"{x:.7f}","Период RK4, с":lambda x:f"{x:.7f}",
    "Ошибка линейного периода, %":lambda x:f"{x:.3f}"})
threshold_rows = []
for tolerance in [0.001,0.01,0.05]:
    threshold = brentq(lambda A:1-T0/exact_period(A)-tolerance,1e-7,np.pi-1e-7)
    threshold_rows.append({"Допустимая ошибка периода, %":100*tolerance,
                           "Предельный начальный угол, °":np.rad2deg(threshold)})
threshold_table = pd.DataFrame(threshold_rows)
show_table(threshold_table,"linear_period_limits",{
    "Допустимая ошибка периода, %":lambda x:f"{x:g}",
    "Предельный начальный угол, °":lambda x:f"{x:.2f}"})

**Рисунок 4 и таблица 2.** Правая панель ограничена углами 0–60°, чтобы было видно пересечение уровня ошибки 1%. Точный и численный периоды совпадают в показанных разрядах. На всём диапазоне 80 амплитуд максимальное расхождение составляет 4.62e-11 с. Это проверка точности RK4 и определения пересечения, а не оценка качества линейной модели.

Период увеличивается с амплитудой: при $60^\circ$ он равен примерно $2.153$ с, а при $120^\circ$ — $2.754$ с. Линейная модель при этом всегда предсказывает $T_0\approx2.006$ с и поэтому занижает период.

В таблице 3 точность линейного прогноза определена как
$$\delta_T(A)=\frac{T(A)-T_0}{T(A)}.$$
Если допустима ошибка **не более 1%**, линейная модель подходит примерно до $A=22.93^\circ$. Для допуска $0.1\%$ граница уменьшается до $7.25^\circ$, а для $5\%$ увеличивается до $51.35^\circ$. Поэтому ответ о «достаточной точности» зависит от заранее выбранного допуска. Например, при $10^\circ$ ошибка периода около $0.19\%$, при $30^\circ$ — уже $1.71\%$.

При малом угле, выраженном в радианах,
$$\frac{T(A)}{T_0}=1+\frac{A^2}{16}+\frac{11A^4}{3072}+O(A^6).$$
Первый поправочный член также объясняет, почему период линейной модели оказывается меньше нелинейного.

## 4. Ошибка модели при начальном угле 60°

Выберу $A=60^\circ$, чтобы расхождение было заметно. Здесь $T(A)=2.152875$ с, а относительная ошибка линейного периода равна 6.819%. Даже небольшая разница периодов постепенно приводит к заметному расхождению фаз.

На следующем рисунке сравниваются **точные решения двух уравнений**. Поэтому показанное расхождение не вызвано шагом численного интегрирования. Для нелинейного решения использована формула через эллиптические функции Якоби:
$$k=\sin(A/2),\quad m=k^2,\quad u=K(m)-\omega_0t,$$
$$\theta_{\rm nl}(t)=2\arcsin\bigl(k\,\operatorname{sn}(u\mid m)\bigr),\qquad
v_{\rm nl}(t)=-2k\omega_0\operatorname{cn}(u\mid m).$$
Функции `sn` и `cn` вычисляет `scipy.special.ellipj`. Эта формула далее служит независимым эталоном для проверки численного решения.

In [ ]:
chosen_amplitude = np.deg2rad(60)
nonlinear_duration = 10*T0
t_compare = np.linspace(0,nonlinear_duration,5001)
y_lin_exact = exact_linear(t_compare,chosen_amplitude)
y_non_exact = exact_nonlinear(t_compare,chosen_amplitude)
fig,axes = plt.subplots(1,2,figsize=(11,4.2),constrained_layout=True)
short = t_compare <= 4*T0
axes[0].plot(t_compare[short]/T0,y_lin_exact[short,0],label="Линейная модель")
axes[0].plot(t_compare[short]/T0,y_non_exact[short,0],label="Нелинейная модель")
axes[0].set(xlabel="Время t/T₀",ylabel="Угол θ, рад",title="Точные решения, первые 4T₀")
axes[0].legend(fontsize=9)
axes[1].plot(t_compare/T0,y_non_exact[:,0]-y_lin_exact[:,0],color="#c34a36")
axes[1].set(xlabel="Время t/T₀",ylabel="Разность углов θнел − θлин, рад",title="Расхождение моделей на интервале 10T₀")
for ax in axes:
    ax.grid(alpha=0.2)
show_figure(fig,"05_model_difference")
fig,axes = plt.subplots(1,2,figsize=(11,4.2),constrained_layout=True)
axes[0].plot(y_lin_exact[:,0],y_lin_exact[:,1],label="Линейная модель")
axes[0].plot(y_non_exact[:,0],y_non_exact[:,1],label="Нелинейная модель")
axes[0].set(xlabel="Угол θ, рад",ylabel="Угловая скорость v, рад/с",title="Фазовые траектории при A=60°")
axes[0].legend(fontsize=9)
U,K = energies(y_non_exact)
axes[1].plot(t_compare/T0,U,label="Потенциальная U")
axes[1].plot(t_compare/T0,K,label="Кинетическая K")
axes[1].plot(t_compare/T0,U+K,label="Полная E",color="#34785a")
axes[1].set(xlabel="Время t/T₀",ylabel="Энергия, Дж",title="Энергии точного нелинейного решения")
axes[1].legend(fontsize=9)
for ax in axes:
    ax.grid(alpha=0.2)
show_figure(fig,"06_nonlinear_phase_energy")

**Рисунок 5.** Линейный маятник постепенно опережает нелинейный. На левой панели показаны первые четыре линейных периода, на правой — разность углов на всём интервале $10T_0$. Разность не возрастает монотонно: сравниваются два ограниченных колебания с разными периодами.

**Рисунок 6.** Фазовые кривые обеих моделей замкнуты, но различаются по форме. Для нелинейной модели использованы энергии $U=mg\ell(1-\cos\theta)$ и $K=m\ell^2v^2/2$. Их сумма остаётся постоянной в точном решении. Различие фазовых кривых показывает, что при большой амплитуде замена $\sin\theta$ на $\theta$ меняет не только период.

## 5. Ошибка численного решения нелинейной модели

Теперь уравнение фиксировано: это нелинейный маятник при $A=60^\circ$. Варьирую только шаг RK4 на одном интервале $[0,10T_0]$: $h=T_0/100$, $T_0/200$, $T_0/400$, $T_0/800$.

В таблице 4 ошибка угла — максимум по узлам относительно точной нелинейной формулы. Ошибка периода равна $|T_{\rm num}-T(A)|$, а ошибка энергии — максимум $100|E_n/E_0-1|$. Оценка порядка $p$ вычисляется по ошибке угла, как в таблице 1.

In [ ]:
nonlinear_resolutions = [100,200,400,800]
nonlinear_rows = []
nonlinear_results = {}
previous = None
for M in nonlinear_resolutions:
    t,y = integrate(chosen_amplitude,nonlinear_duration,10*M)
    h = t[1]-t[0]
    nonlinear_results[M] = t,y
    error = np.max(np.abs(y[:,0]-exact_nonlinear(t,chosen_amplitude)[:,0]))
    U,K = energies(y)
    p = np.nan if previous is None else np.log2(previous/error)
    nonlinear_rows.append({"Шаг h, с":h,"Ошибка угла, рад":error,
                           "Ошибка периода, с":abs(numerical_period(chosen_amplitude,h)-exact_period(chosen_amplitude)),
                           "Ошибка энергии, %":100*np.max(np.abs((U+K)/(U[0]+K[0])-1)),
                           "Оценка порядка p":p})
    previous = error
nonlinear_table = pd.DataFrame(nonlinear_rows)
show_table(nonlinear_table,"nonlinear_convergence",{
    "Шаг h, с":lambda x:f"{x:.7f}","Ошибка угла, рад":lambda x:f"{x:.3e}",
    "Ошибка периода, с":lambda x:f"{x:.3e}","Ошибка энергии, %":lambda x:f"{x:.3e}",
    "Оценка порядка p":lambda x:"—" if np.isnan(x) else f"{x:.3f}"})
fig,axes = plt.subplots(1,2,figsize=(11,4.2),constrained_layout=True)
for M,(t,y) in nonlinear_results.items():
    error = np.abs(y[:,0]-exact_nonlinear(t,chosen_amplitude)[:,0])
    axes[0].semilogy(t/T0,np.where(error>0,error,np.nan),label=f"h = T₀/{M}")
axes[0].set(xlabel="Время t/T₀",ylabel="Абсолютная ошибка угла, рад",title="Ошибка RK4 при одном уравнении")
hs = nonlinear_table["Шаг h, с"].to_numpy()
es = nonlinear_table["Ошибка угла, рад"].to_numpy()
axes[1].loglog(hs,es,"o-",label="Вычисленная ошибка")
axes[1].loglog(hs,es[-1]*(hs/hs[-1])**4,"k--",label="Зависимость пропорционально h⁴")
axes[1].set(xlabel="Шаг интегрирования h, с",ylabel="Максимальная ошибка угла, рад",title="Сходимость при уменьшении шага")
axes[1].set_xticks(hs,labels=[f"{h:.5f}" for h in hs])
axes[1].tick_params(axis="x",which="minor",labelbottom=False,bottom=False)
for ax in axes:
    ax.grid(alpha=0.2,which="both")
    ax.legend(fontsize=8)
show_figure(fig,"07_nonlinear_convergence")

**Таблица 4 и рисунок 7.** При уменьшении шага все три измеряемые ошибки уменьшаются. На последней паре сеток оценка порядка по углу равна 3.948 и приближается к четырём. Она не обязана точно равняться четырём на конечных шагах: в ошибке остаются члены более высоких порядков, а положение пересечения внутри шага меняется.

На самом мелком шаге максимальная ошибка угла составляет 1.27e-09 рад. Это намного меньше расхождения точных моделей на рисунке 5. Поэтому уточнение интегрирования не устранит запаздывание нелинейного маятника относительно линейного: оно обусловлено разницей уравнений.

## 6. Два маятника и фазовая плоскость

На первой анимации линейная и нелинейная модели показаны с наложением при одинаковых $A=60^\circ$, длине и начальной скорости. Слева отображается положение грузов, справа — текущие состояния на фазовой плоскости. Точки справа оставляют следы. Обе панели используют один массив моментов времени.

Для нелинейной модели используется RK4 с шагом не более $0.002$ с, для линейной — точная формула. Между узлами выполнена линейная интерполяция только для отображения кадров. При оценке ошибок в таблицах она не применяется.

In [ ]:
def render_animation(animation, filename, fps):
    animation._fig.canvas.draw()
    animation._fig.set_layout_engine(None)
    gif_path = OUTPUT/"media"/f"{filename}.gif"
    if writers.is_available("ffmpeg"):
        mp4_path = OUTPUT/"media"/f"{filename}.mp4"
        animation.save(mp4_path,writer=FFMpegWriter(fps=fps,codec="libx264",
                       extra_args=["-pix_fmt","yuv420p","-profile:v","baseline","-vf","pad=ceil(iw/2)*2:ceil(ih/2)*2","-movflags","+faststart"]),dpi=100)
        subprocess.run(["ffmpeg","-v","error","-y","-i",str(mp4_path),"-filter_complex",
                        "fps=12,scale=760:-2:flags=lanczos,split[a][b];[a]palettegen=max_colors=128:stats_mode=diff[p];[b][p]paletteuse=dither=none",
                        "-loop","0",str(gif_path)],check=True)
    else:
        animation.save(gif_path,writer=PillowWriter(fps=fps),dpi=85)
    return Image(data=gif_path.read_bytes(),format="gif")


def animation_states(amplitude, frames_t, ell=length):
    t,y = integrate(amplitude,frames_t[-1],int(np.ceil(frames_t[-1]/0.002)),ell=ell)
    return np.column_stack([np.interp(frames_t,t,y[:,j]) for j in range(2)])


fps = 15
comparison_end = 6*T0
comparison_times = np.linspace(0,comparison_end,round(comparison_end*fps)+1)
comparison_linear = exact_linear(comparison_times,chosen_amplitude)
comparison_nonlinear = animation_states(chosen_amplitude,comparison_times)
fig,axes = plt.subplots(1,2,figsize=(10,4.5),constrained_layout=True)
axes[0].set(xlim=(-1.15,1.15),ylim=(-1.15,0.25),xlabel="x, м",ylabel="y, м",title="Положение грузов; A=60°, длина 1 м")
axes[0].set_aspect("equal")
axes[0].plot(0,0,"ko",ms=4)
axes[1].set(xlim=(-1.2,1.2),ylim=(-3.6,3.6),xlabel="Угол θ, рад",ylabel="Угловая скорость v, рад/с",title="Состояния и следы на фазовой плоскости")
comparison_artists = []
for label,color in [("Линейная модель","#2166ac"),("Нелинейная модель","#c34a36")]:
    rod, = axes[0].plot([],[],"o-",lw=2,ms=7,color=color,label=label)
    trace, = axes[1].plot([],[],lw=1.3,color=color,label=label)
    point, = axes[1].plot([],[],"o",ms=6,color=color)
    comparison_artists.append((rod,trace,point))
for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8,loc="upper right")
comparison_clock = fig.suptitle("Общее время t=0 с")


def update_comparison(i):
    result = []
    for states,(rod,trace,point) in zip([comparison_linear,comparison_nonlinear],comparison_artists):
        angle = states[i,0]
        rod.set_data([0,length*np.sin(angle)],[0,-length*np.cos(angle)])
        trace.set_data(states[:i+1,0],states[:i+1,1])
        point.set_data([states[i,0]],[states[i,1]])
        result.extend([rod,trace,point])
    comparison_clock.set_text(f"Общее время t={comparison_times[i]:.2f} с")
    return result+[comparison_clock]


comparison_animation = FuncAnimation(fig,update_comparison,frames=len(comparison_times),interval=1000/fps,blit=False)
display(render_animation(comparison_animation,"01_models_phase",fps))
plt.close(fig)

**Анимация 1.** Нелинейный груз постепенно запаздывает. На фазовой плоскости видно различие формы замкнутых траекторий; различие положения текущих точек показывает накопившуюся разность фаз. Начальное наложение двух грузов соответствует одинаковым начальным условиям.

## 7. Совмещённый ансамбль маятников разной длины

Задам десять частот линейных моделей $\omega_i=\omega_*n_i$, где $\omega_*=0.25$ рад/с, $n_i=10,\ldots,19$. Тогда
$$\ell_i=\frac{g}{\omega_i^2}.$$
Все маятники отпускаются из одного угла $20^\circ$ с нулевой скоростью. Они не взаимодействуют и изображаются на **одной координатной плоскости с общим подвесом**. Длины показаны в одном физическом масштабе.

Движение рассчитывается по нелинейному уравнению. Поскольку амплитуды одинаковы, поправка к периоду у всех одинакова:
$$F(A)=\frac{2K(\sin^2(A/2))}{\pi},\qquad \Omega_i=\frac{\omega_*n_i}{F(A)}.$$
Фактические частоты $\Omega_i$ также соизмеримы. Через
$$T_*=\frac{2\pi F(A)}{\omega_*}$$
каждый маятник совершит целое число $n_i$ полных колебаний и вернётся в начальное состояние. В данном случае $T_*=25.3255$ с. Анимация охватывает именно этот интервал.

In [ ]:
harmonics = np.arange(10,20)
base_frequency = 0.25
ensemble_lengths = g/(base_frequency*harmonics)**2
common_amplitude = np.deg2rad(20)
period_factor = 2*ellipk(np.sin(common_amplitude/2)**2)/np.pi
sync_time = 2*np.pi*period_factor/base_frequency
length_times = np.linspace(0,sync_time,round(sync_time*fps)+1)
length_states = [animation_states(common_amplitude,length_times,ell) for ell in ensemble_lengths]
length_table = pd.DataFrame({"Число колебаний за T*":harmonics,"Длина, м":ensemble_lengths,
                             "Период, с":exact_period(common_amplitude,ensemble_lengths)})
show_table(length_table,"length_ensemble",{"Длина, м":lambda x:f"{x:.4f}","Период, с":lambda x:f"{x:.4f}"})
fig,ax = plt.subplots(figsize=(7.5,5.8),constrained_layout=True)
ax.set(xlim=(-0.72,0.72),ylim=(-1.7,0.18),xlabel="x, м",ylabel="y, м",
       title="Десять маятников разной длины; начальный угол 20°")
ax.set_aspect("equal")
ax.grid(alpha=0.2)
ax.plot(0,0,"ko",ms=5,zorder=5)
length_rods = []
for n,ell,color in zip(harmonics,ensemble_lengths,plt.cm.viridis(np.linspace(0.04,0.9,10))):
    rod, = ax.plot([],[],"o-",lw=1.6,ms=6,color=color,alpha=0.9,label=f"n={n}, длина {ell:.3f} м")
    length_rods.append(rod)
ax.legend(loc="center left",bbox_to_anchor=(1.02,0.5),fontsize=8,frameon=False)
length_clock = ax.text(0.02,0.96,"",transform=ax.transAxes,va="top",fontsize=10)


def update_lengths(i):
    for rod,ell,states in zip(length_rods,ensemble_lengths,length_states):
        angle = states[i,0]
        rod.set_data([0,ell*np.sin(angle)],[0,-ell*np.cos(angle)])
    length_clock.set_text(f"t={length_times[i]:.2f} с из {sync_time:.2f} с")
    return length_rods+[length_clock]


length_animation = FuncAnimation(fig,update_lengths,frames=len(length_times),interval=1000/fps,blit=False)
display(render_animation(length_animation,"02_lengths_overlay",fps))
plt.close(fig)

**Таблица 5 и анимация 2.** Длинные маятники колеблются медленнее коротких. В промежуточные моменты грузы расходятся по фазе, но в конце интервала снова оказываются на одном начальном луче. Это возвращение следует из выбранного соотношения частот, а не из наличия связи между маятниками. Цвет одного маятника постоянен во всех кадрах.

## 8. Совмещённый ансамбль одинаковой длины при разных амплитудах

Теперь длины одинаковы: $\ell=1$ м. Начальные углы составляют $10^\circ,20^\circ,\ldots,100^\circ$, скорости равны нулю. Все десять маятников снова показаны **на одной плоскости с общим подвесом**, без отдельных панелей и искусственных смещений осей.

В линейной модели период не зависел бы от начального угла. В нелинейной модели маятники с большими амплитудами колеблются медленнее. Например, для $10^\circ$ период около $2.010$ с, а для $100^\circ$ — $2.472$ с. На интервале $12T_0$ это различие приводит к заметному расхождению фаз.

In [ ]:
ensemble_amplitudes_deg = np.arange(10,101,10)
ensemble_amplitudes = np.deg2rad(ensemble_amplitudes_deg)
amplitude_end = 12*T0
amplitude_times = np.linspace(0,amplitude_end,round(amplitude_end*fps)+1)
amplitude_states = [animation_states(A,amplitude_times) for A in ensemble_amplitudes]
amplitude_table = pd.DataFrame({"Начальный угол, °":ensemble_amplitudes_deg,
                                "Нелинейный период, с":exact_period(ensemble_amplitudes)})
show_table(amplitude_table,"amplitude_ensemble",{"Нелинейный период, с":lambda x:f"{x:.4f}"})
fig,ax = plt.subplots(figsize=(7.5,5.5),constrained_layout=True)
ax.set(xlim=(-1.12,1.12),ylim=(-1.15,0.35),xlabel="x, м",ylabel="y, м",
       title="Десять маятников длиной 1 м; разные начальные углы")
ax.set_aspect("equal")
ax.grid(alpha=0.2)
ax.plot(0,0,"ko",ms=5,zorder=5)
amplitude_rods = []
for deg,color in zip(ensemble_amplitudes_deg,plt.cm.plasma(np.linspace(0.04,0.9,10))):
    rod, = ax.plot([],[],"o-",lw=1.5,ms=6,color=color,alpha=0.9,label=f"Начальный угол {deg}°")
    amplitude_rods.append(rod)
ax.legend(loc="center left",bbox_to_anchor=(1.02,0.5),fontsize=8,frameon=False)
amplitude_clock = ax.text(0.02,0.96,"",transform=ax.transAxes,va="top",fontsize=10)


def update_amplitudes(i):
    for rod,states in zip(amplitude_rods,amplitude_states):
        angle = states[i,0]
        rod.set_data([0,length*np.sin(angle)],[0,-length*np.cos(angle)])
    amplitude_clock.set_text(f"Общее время t={amplitude_times[i]:.2f} с")
    return amplitude_rods+[amplitude_clock]


amplitude_animation = FuncAnimation(fig,update_amplitudes,frames=len(amplitude_times),interval=1000/fps,blit=False)
display(render_animation(amplitude_animation,"03_amplitudes_overlay",fps))
plt.close(fig)

### Проверка вычислений

Перед выводами проверю согласованность результатов: совпадение периодов с точной формулой, порядки сходимости, сохранение энергии точного нелинейного решения и возвращение ансамбля разной длины. Эти проверки выполняются следующей ячейкой.

In [ ]:
assert abs(T0-2.0060666807106475)<1e-12
assert np.max(np.abs(periods_numeric-periods_exact))<1e-7
assert nonlinear_table["Оценка порядка p"].iloc[-1]>3.8
for method,expected_order in [("Эйлер",1.0),("RK4",4.0)]:
    final_order = linear_table[linear_table["Метод"]==method]["Оценка порядка p"].iloc[-1]
    assert abs(final_order-expected_order)<0.1
for A in [0.1,np.deg2rad(60),np.deg2rad(100)]:
    y = exact_nonlinear(np.linspace(0,10,501),A)
    U,K = energies(y)
    assert np.ptp(U+K)<1e-12
for states in length_states:
    assert abs(states[-1,0]-common_amplitude)<1e-7
    assert abs(states[-1,1])<1e-6
print("Проверены период, сходимость, энергия точного нелинейного решения и возвращение ансамбля разной длины.")

**Таблица 6 и анимация 3.** Цвета соответствуют начальным углам из легенды. При одинаковой длине грузы движутся по одной окружности, но с разными амплитудами и периодами. В момент запуска все маятники находятся в крайних положениях. По мере движения большие амплитуды приводят к запаздыванию; все маятники уже не проходят нижнее положение одновременно.

## 9. Выводы

Проверка на линейном маятнике показала ожидаемую сходимость: метод Эйлера имеет первый порядок, RK4 — четвёртый. При этом явный Эйлер увеличивает полную энергию и амплитуду колебаний. Уменьшение шага снижает ошибку на заданном интервале, но не устраняет рост энергии при продолжительном расчёте с фиксированным шагом.

В нелинейной модели период зависит от амплитуды. Для выбранной длины линейный период равен $2.006067$ с. При $60^\circ$ нелинейный период составляет $2.152875$ с, поэтому линейная модель занижает его примерно на $6.819\%$. Если требуется ошибка периода не более $1\%$, линейное приближение подходит для начальных углов примерно до $22.93^\circ$. Эта граница относится к периоду; на длинном интервале даже небольшая ошибка периода может дать заметную разность фаз.

Сгущение сетки RK4 уменьшает ошибку решения **одного и того же** нелинейного уравнения и приближает оценку порядка к четырём. Оно не устраняет различие между линейной и нелинейной моделями. Анимации показывают оба эффекта изменения периода: из-за разной длины и из-за разной начальной амплитуды. Все ансамбли изображены с наложением на общей плоскости.